# 📑 NPL Securitization Report
 
## 🔗 Links
- **[BRD - NPL Securitization Report](https://cepalhellas.atlassian.net/wiki/spaces/DBTC/pages/253132810/BRD+-+NPL+Securitisation+Reports+to+Cloud)**  
  Comprehensive documentation outlining the business requirements for this report.
 
- **[Story Ticket](https://cepalhellas.atlassian.net/browse/DBTC-400)**  
  Details about the story ticket, including tasks and progress.
 
- **[UAT Findings Ticket](https://cepalhellas.atlassian.net/browse/ASK-1163)**  
  A list of findings from UAT.
 
---

# Install Dependencies

In [0]:
%pip install openpyxl
%pip install xlsxwriter
dbutils.library.restartPython()

In [0]:
import os
import sys
from pytz import timezone
from datetime import datetime
import json


try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = "/Workspace" + notebook_path.rsplit("/NPLSecuritization/", 1)[0]
    sys.path.insert(0, os.path.join(repo_root, "aws_utils"))
    sys.path.insert(0, os.path.join(repo_root, "01-utils"))
except Exception:
    sys.path.append("../../aws_utils")
    sys.path.append("../../01-utils")

from aws_utils import get_env_prefix, get_env_fullname

env_prefix = get_env_prefix()

In [0]:
from typing import Any
from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
    regexp_replace
)

import os

env = env_prefix

from typing import Optional
import delta  # type: ignore
from pyspark.sql import SparkSession
from io import BytesIO
import pandas as pd
import json
from datetime import datetime
from pytz import timezone

In [0]:
athens_tz = timezone('Europe/Athens')
try:    
    report_args = dbutils.widgets.get("report_args")
    report_args = report_args.replace("'", '"')
except Exception as e:
    raise ValueError("Missing required input: report_args")
# try:    
#     report_name = dbutils.widgets.get("report_name")
# except Exception as e:
#     raise ValueError("Missing required input: report_name")
# print(f"Report {report_name} is started")

run_date_raw = dbutils.widgets.get("run_date").strip()
if run_date_raw:
    run_date = run_date_raw
    run_date_str = run_date.replace('-', '')
    referenceDate = run_date
else:
    run_date = datetime.now(athens_tz).strftime('%Y-%m-%d')
    run_date_str = datetime.now(athens_tz).strftime('%Y%m%d')
    referenceDate = spark.sql(f"SELECT CurrentMonthLastWorkingDate FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate = last_day(add_months(to_date('{run_date}'), -1))").first()["CurrentMonthLastWorkingDate"]
    referenceDate = referenceDate.strftime('%Y-%m-%d')
now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')



# Read JSON
json_data = json.loads(report_args)
# Access the values and store them into variables
# if "portfolio_name" not in json_data: raise KeyError("'portfolio_name' key is missing in the report_args JSON data.")
if "output_file_prefix" not in json_data: raise KeyError("'output_file_prefix' key is missing in the report_args JSON data.")
if "report_output" not in json_data: raise KeyError("'report_output' key is missing in the report_args JSON data.")
# portfolioName = "".join(json_data["portfolio_name"])
output_file_prefix = "".join(json_data["output_file_prefix"])
report_output = "".join(json_data["report_output"])
# Resolve full export path from SSM if relative (same pattern as MegaDataTape)
if not report_output.startswith("/Volumes"):
    from aws_utils import get_scope_key_names, get_aws_credentials, connect_to_aws_and_access_ssm
    import boto3
    _secrets_scope = "aws-databricks"
    _aws_keys = get_scope_key_names(_secrets_scope)
    _aws_key = _aws_keys[0] if _aws_keys else None
    _aws_secret_key = get_aws_credentials(_secrets_scope, _aws_key)
    _ssm_client = boto3.client("ssm", region_name="eu-central-1",
                               aws_access_key_id=_aws_key,
                               aws_secret_access_key=_aws_secret_key)
    export_base_path = connect_to_aws_and_access_ssm(
        "/reports/exports/s3_main_path", _ssm_client, _aws_secret_key
    ).replace("envprefix", env)
    report_output = export_base_path + report_output
# print(f"Report {report_name} is running with the following arguments: report_args = {report_args}, report_name = {report_name}, portfolio_name = {portfolioName}, output_file_prefix = {output_file_prefix}, run_date = {run_date}")
print(f"now is: {now_datetime_str}")

In [0]:
# Specify portfolios in scope as defined in the BRD
portfolios_in_scope = (
    'Cosmos Retail',
    'Cosmos Wholesale',
    'Orion X Securitisation DAC',
    'Galaxy II Funding DAC',
    'Galaxy IV Funding DAC',
)

# Import Tables

## Import Tables - NPL Securitization Exposures

In [0]:
dparty = spark.sql(f"""
                   SELECT PortfolioName,
                          BusinessDate AS ReferenceDate,
                          CDI
                   FROM `{env_prefix}-cdh`.derivative.dparty
                   WHERE BusinessDate =  {referenceDate!r}
                   AND sourcesystem='profits'
                   --AND PortfolioName IN ({', '.join([f"'{p}'" for p in portfolios_in_scope])})
                   AND CDI IS NOT NULL
                   """
)
dparty.createOrReplaceTempView("dparty")  

dexposure = spark.sql(f"""
                      SELECT PortfolioName, 
                             BusinessDate AS ReferenceDate, 
                             ExposureKey,
                             AgreementKey,
                             PartyTaxRegistrationNumber,
                             CDI, 
                             OutstandingTotalBalance, 
                             GBV,
                             Currency, 
                             ExposureStatus, 
                             CAST(HasActiveSettlementFlag AS INT) AS HasActiveSettlementFlag, 
                             ExposureRestructureType,
                             ProductKey,
                             IsActiveFlag,
                             PEERestructureTypeCode
                      FROM `{env_prefix}-cdh`.derivative.get_all_exposures_by_business_date(DATE({referenceDate!r}))
                      WHERE sourcesystem = 'profits'
                      --AND PortfolioName IN ({', '.join([f"'{p}'" for p in portfolios_in_scope])})
                      """)

dexposure.createOrReplaceTempView("dexposure")

exposure = spark.sql(f"""
                     SELECT ExposureKey, Bucket, CDI, AgreementKey
                     FROM `{env_prefix}-cdh`.primary.exposure
                     WHERE BusinessDate = {referenceDate!r}
                     AND SourceSystem = 'profits'
                     AND PartyRelationshipType = 'Primary Owner'
                     """)

exposure.createOrReplaceTempView("exposure")

product = spark.sql(f"""
                    SELECT BusinessDate AS ReferenceDate, 
                           ProductKey,
                           ProductDescription
                    FROM `{env_prefix}-cdh`.primary.product
                    WHERE BusinessDate = {referenceDate!r} 
                    AND SourceSystem = 'profits' 
                    """)

product.createOrReplaceTempView("product") 

underlaw = spark.sql(f"""
                      SELECT BusinessDate AS ReferenceDate, 
                             PortfolioName,
                             CDI, 
                             UnderLawType, 
                             CalcLawStatusCode, 
                             CourtDate,
                             InLawFlag
                      FROM `{env_prefix}-cdh`.derivative.dunderlaw
                      WHERE BusinessDate = {referenceDate!r} 
                      --AND PortfolioName IN ({', '.join([f"'{p}'" for p in portfolios_in_scope])})
                      AND UnderLawType = 'N3869'
                      """)

underlaw.createOrReplaceTempView("underlaw")
 
debtor = spark.sql(f"""
                     SELECT PortfolioName, 
                            BusinessDate AS ReferenceDate, 
                            CDI,
                            OperationalLegalStatusLevel3Code
                     FROM `{env_prefix}-cdh`.derivative.dlegalprocessstatusdebtor
                     WHERE 1=1--PortfolioName IN ({', '.join([f"'{p}'" for p in portfolios_in_scope])})
                     AND BusinessDate = {referenceDate!r}
                     AND OriginSourceSystem = 'qca' 
                     """)

debtor.createOrReplaceTempView("debtor") 

borrower = spark.sql(f"""
                      SELECT DISTINCT DebtorCode, BPApproach
                      FROM `{env_prefix}-cdh`.staging.dwh_stg_biteam_whl_glx_cs_haps_targets_per_borrower_vw
                      WHERE DebtorCode IS NOT NULL 
                      """)

borrower.createOrReplaceTempView("borrower") 

exchange_rates = spark.sql(f"""
                     SELECT BusinessDate, 
                     OriginalCurrency,
                     ExchangeRate                                       
                     FROM `{env_prefix}-cdh`.reference.exchangerates
                     WHERE BusinessDate = {referenceDate!r} 
                     """)
exchange_rates.createOrReplaceTempView("exchange_rates")


## Import Tables - NPL Securitization Properties

In [0]:
drealestate = spark.sql(f"""
                        SELECT DISTINCT RealEstateKey, 
                                        RealEstateCepalKey
                        FROM `{env_prefix}-cdh`.derivative.drealestate
                        WHERE BusinessDate = {referenceDate!r} 
                      """)
drealestate.createOrReplaceTempView("drealestate")

dcollateral = spark.sql(f"""
                        SELECT DISTINCT OwnershipKey, 
                                        DCollateralKey, 
                                        AgreementKey, 
                                        CollateralType, 
                                        RealEstateKey, 
                                        PortfolioName
                        FROM `{env_prefix}-cdh`.derivative.dcollateral
                        WHERE BusinessDate = {referenceDate!r}
                        AND PortfolioName IN ({', '.join([f"'{p}'" for p in portfolios_in_scope])})
                      """)

dcollateral.createOrReplaceTempView("dcollateral")

# dexposure = spark.sql(f"""
#                       SELECT DISTINCT AgreementKey, 
#                                       CDI, 
#                                       BusinessDate AS ReferenceDate, 
#                                       PortfolioName, 
#                                       ExposureKey
#                       FROM `{env_prefix}-cdh`.derivative.get_all_exposures_by_business_date(DATE({referenceDate!r}))
#                       WHERE sourcesystem = 'profits'
#                       AND PortfolioName IN ({', '.join([f"'{p}'" for p in portfolios_in_scope])})
#                       AND BusinessDate = {referenceDate!r}
#                       """)

#dexposure.createOrReplaceTempView("dexposure")

# dparty = spark.sql(f"""
#                    SELECT * 
#                    FROM `{env_prefix}-cdh`.derivative.dparty
#                    WHERE BusinessDate = {referenceDate!r} 
#                       """)

# dparty.createOrReplaceTempView("dparty")

realestatevaluation = spark.sql(f"""
                                SELECT RealEstateKey,
                                  RealEstateValuationType1,
                                  RealEstateValuationType2,
                                  ValuationDate,
                                  MarketValue,
                                  RealEstateValuationKey,
                                  IsActiveFlag,
                                  RealEstateValuatorKey,
                                  BusinessDate
                                FROM `{env_prefix}-cdh`.primary.realestatevaluation
                                WHERE BusinessDate = {referenceDate!r} 
                                AND SourceSystem = 'thesis'
                               """)

realestatevaluation.createOrReplaceTempView("realestatevaluation")

auction = spark.sql(f""" 
                    SELECT Cepal_Key_Final, 
                    AuctionDate, 
                    sum(FirstOfferPrice) FirstOfferPrice, 
                    sum(FinalPrice) FinalPrice,
                     `Κατηγορία3_active`
                    FROM (
                            SELECT DISTINCT AuctionObJID, 
                                            FirstOfferPrice, 
                                            FinalPrice,
                                            Cepal_Key_Final, 
                                            AuctionDate, 
                                             `Κατηγορία3_active`,
                                            rank() over (partition by Cepal_Key_Final order by AuctionDate desc, BusinessDate desc) as rank
                            FROM `p-cdh-delivery`.tia.tia_historical
                            WHERE AuctionDate <= {referenceDate!r}
                            AND BusinessDate <= {referenceDate!r}
                    ) WHERE rank = 1
                    GROUP BY Cepal_Key_Final, AuctionDate, `Κατηγορία3_active`
                    """)


auction.createOrReplaceTempView("auction")

dregexposurefields = spark.sql(f"""
                   SELECT AgreementKey, 
                          PortfolioName, 
                          ExposureKey, 
                          CollateralAmountRRE
                   FROM `{env_prefix}-cdh`.derivative.dregexposurefields
                   WHERE BusinessDate = {referenceDate!r} 
                   AND PortfolioName IN ({', '.join([f"'{p}'" for p in portfolios_in_scope])})
                      """)

dregexposurefields.createOrReplaceTempView("dregexposurefields")

# Pre-processing

## Pre-processing - NPL Securitization Exposures

In [0]:
# Calculate first day of the month for reference_date
first_of_month = datetime.strptime(referenceDate, '%Y-%m-%d').replace(day=1).strftime('%Y-%m-%d')

# Keep entries CDIs with GBV > 0€ at least one day in the reference month of the report
dexposure_proc = spark.sql(f"""
    SELECT dexp.*
    FROM dexposure dexp
    INNER JOIN (
       SELECT DISTINCT ExposureKey, PortfolioName
        FROM `{env_prefix}-cdh`.derivative.dexposure 
        WHERE BusinessDate BETWEEN {first_of_month!r} AND {referenceDate!r}
        AND GBV > 0 AND IsActiveFlag = TRUE
    ) sub
    ON dexp.ExposureKey = sub.ExposureKey AND dexp.PortfolioName = sub.PortfolioName
""")

dexposure_proc.createOrReplaceTempView("dexposure_proc") 

In [0]:
# join all previous dataframes where the dexposure_proc is on the felt part of the join
joined_table_exp = spark.sql(f"""
    SELECT 
        dexp.*, 
        exp.Bucket AS SourceBucket,
        ER.ExchangeRate,
        -- create column GBVEuro
        CASE 
            WHEN dexp.Currency = 'EUR' THEN dexp.GBV
            ELSE CAST(dexp.GBV / COALESCE(ER.ExchangeRate, 1)  AS DECIMAL(19,2))
        END AS GBVEuro,
        -- create column ConfiscationStatus
        CASE
            WHEN debtor.OperationalLegalStatusLevel3Code < 411 THEN 'Pre-Confiscation'
            WHEN debtor.OperationalLegalStatusLevel3Code > 331 THEN 'Post-Confiscation' 
            ELSE 'N/A'
        END AS ConfiscationStatus,
        -- create column LegalStatus
        CASE
            WHEN debtor.OperationalLegalStatusLevel3Code IN (311,312,321,331) THEN 'Payment Order'
            WHEN debtor.OperationalLegalStatusLevel3Code IN (211,221,231,232,241,242,251,261) THEN 'Extrajudicial notice'
            WHEN debtor.OperationalLegalStatusLevel3Code > 331 THEN "Seizure"
            ELSE 'No Legal Action'
        END AS LegalStatus,
        product.ProductDescription,
        -- create column L3869_DetailedStatus
        CASE
            WHEN under.CalcLawStatusCode = 'Pending Court Decision' AND under.CourtDate > {referenceDate!r} THEN 'Future Court date'
            WHEN under.CalcLawStatusCode = 'Pending Court Decision' AND (under.CourtDate <= {referenceDate!r} or under.CourtDate is null) THEN 'Temp - Pending Temporary Order'
            WHEN under.CalcLawStatusCode = 'Positive Court Decision' THEN 'Ruled - Accepted'
            WHEN under.InLawFlag = 0 THEN 'Ruled Rejected'
            ELSE 'N/A'
        END AS L3869_DetailedStatus,
        -- create column IsActiveModification
        CASE 
            WHEN ((dexp.ExposureRestructureType = 'Restructure' OR dexp.HasActiveSettlementFlag = 1) AND dexp.ExposureStatus IN ('Normal', 'Overdue')) THEN 1
            ELSE 0
        END AS IsActiveModification,
        -- create column TopBookBorrowerFlag
        CASE WHEN borr.BPApproach = 'Individual' THEN 1 ELSE 0 END AS TopBookBorrowerFlag,
        -- create column ModificationType
        CASE
            WHEN dexp.ExposureRestructureType = 'Restructure' OR dexp.HasActiveSettlementFlag = 1 THEN 
            -- WHEN IsRestructuredFlag = 1 OR dexp.HasActiveSettlementFlag = 1 THEN 
                CASE
                    WHEN dexp.ProductKey LIKE '%8000' THEN 'G1'
                    WHEN dexp.ProductKey LIKE '%8050' THEN 'G1 Su'
                    WHEN dexp.ProductKey LIKE '%8100' THEN 'G2'
                    WHEN dexp.ProductKey LIKE '%8150' THEN 'G3'
                    WHEN dexp.ProductKey LIKE '%8200' THEN 'G3 SU'
                    WHEN dexp.ProductKey LIKE '%8250' THEN 'G4'
                    WHEN dexp.ProductKey LIKE '%8300' THEN 'G4 SU'
                    WHEN dexp.ProductKey LIKE '%8350' THEN 'G5'
                    WHEN dexp.ProductKey LIKE '%8450' THEN 'G6'
                    WHEN dexp.ProductKey LIKE '%95000' THEN 'S1'
                    WHEN dexp.ProductKey LIKE '%95050' THEN 'S1 Su'
                    WHEN dexp.ProductKey LIKE '%95100' THEN 'S2'
                    WHEN dexp.ProductKey LIKE '%95150' THEN 'S3'
                    WHEN dexp.ProductKey LIKE '%95200' THEN 'S3 SU'
                    WHEN dexp.ProductKey LIKE '%95250' THEN 'S4'
                    WHEN dexp.ProductKey LIKE '%95300' THEN 'S4 Su'
                    WHEN dexp.ProductKey LIKE '%95350' THEN 'S5'
                    WHEN dexp.ProductKey LIKE '%95400' THEN 'S5 Su'
                    WHEN dexp.ProductKey LIKE '%95450' THEN 'S6'
                    WHEN dexp.ProductKey LIKE '%8500' THEN 'G8'
                    WHEN dexp.ProductKey LIKE '%8501' THEN 'G8'
                    WHEN dexp.ProductKey LIKE '%8502' THEN 'G8'
                    WHEN dexp.ProductKey LIKE '%4015' THEN 'L.3869'
                    WHEN dexp.ProductKey LIKE '%4016' THEN 'L.3869'
                    WHEN dexp.ProductKey LIKE '%6078' THEN 'L.3869'
                    WHEN dexp.ProductKey LIKE '%6079' THEN 'L.3869'
                    WHEN dexp.ProductKey LIKE '%7362' THEN 'L.3869'
                    WHEN dexp.ProductKey LIKE '%7363' THEN 'L.3869'
                    WHEN dexp.ProductKey LIKE '%40000' THEN 'L.3869'
                    WHEN dexp.ProductKey LIKE '%40001' THEN 'L.3869'
                    ELSE 'N/A'
                END
            WHEN dexp.ExposureRestructureType != 'Restructure' AND dexp.HasActiveSettlementFlag = 0 THEN 'Unmodified'
            ELSE 'N/A'
        END AS ModificationType,
        -- create column Previous_LegalStatus
        CASE  
            WHEN deb_prev.OperationalLegalStatusLevel3Code IN (311,312,321,331) THEN 'Payment Order'
            WHEN deb_prev.OperationalLegalStatusLevel3Code IN (211,221,231,232,241,242,251,261) THEN 'Extrajudicial notice'
            WHEN deb_prev.OperationalLegalStatusLevel3Code > 331 THEN 'Seizure'
            ELSE 'No Legal Action'
        END AS Previous_LegalStatus
    FROM dexposure_proc dexp
    LEFT JOIN dparty
        ON dexp.CDI = dparty.CDI
        AND dexp.PortfolioName = dparty.PortfolioName
      --  AND dparty.CDI IS NOT NULL AND dparty.SourceSystem = 'profits'
    LEFT JOIN exposure exp
        ON dexp.AgreementKey = exp.AgreementKey
        AND dexp.CDI = exp.CDI
        AND dexp.ExposureKey = exp.ExposureKey
    LEFT JOIN debtor
        ON dexp.PortfolioName = debtor.PortfolioName
        AND dexp.CDI = debtor.CDI
        AND dexp.ReferenceDate = debtor.ReferenceDate
    LEFT JOIN underlaw under
        ON dexp.CDI = under.CDI
        AND dexp.PortfolioName = under.PortfolioName
    LEFT JOIN borrower borr
        ON dexp.CDI = borr.DebtorCode
    LEFT JOIN product
        ON dexp.ProductKey = product.ProductKey
    LEFT JOIN (
    SELECT PortfolioName, CDI, OperationalLegalStatusLevel3Code
    FROM `{env_prefix}-cdh`.derivative.dlegalprocessstatusdebtor
    WHERE OriginSourceSystem = 'qca'
    AND PortfolioName IN ({', '.join([f"'{p}'" for p in portfolios_in_scope])})
    AND OriginSourceSystem = 'qca'
    AND BusinessDate = (
        SELECT MAX(BusinessDate)
        FROM `{env_prefix}-cdh`.derivative.dlegalprocessstatusdebtor
        WHERE BusinessDate < date_trunc('month', DATE({referenceDate!r}))
        AND dayofweek(BusinessDate) BETWEEN 2 AND 6
    )
) deb_prev
    ON dexp.PortfolioName = deb_prev.PortfolioName
    AND dexp.CDI = deb_prev.CDI
    LEFT JOIN exchange_rates ER ON dexp.Currency = ER.OriginalCurrency
""")

joined_table_exp.createOrReplaceTempView("joined_table_exp")

## Pre-processing - NPL Securitization Properties

In [0]:
latest_valuation = spark.sql(f"""
                                SELECT * 
                                FROM(
                                  SELECT * , rank() over (partition by RealEstateKey order by ValuationDate desc) rnk 
                                  FROM realestatevaluation
                                )
                                WHERE rnk = 1
                               """)

latest_valuation.createOrReplaceTempView("latest_valuation")

In [0]:
joined_table_props = spark.sql(f"""
        SELECT BusinessDate, 
                CDI, 
                RealEstateCepalKey, 
                PortfolioName,
                cast(max(LatestValuationDate) AS date) AS LatestValuationDate,
                max(AuctionFirstOfferPrice) AS AuctionFirstOfferPrice, 
                max(AuctionFinalPrice) AS AuctionFinalPrice, 
                max(AuctionStatusCode) AS AuctionStatusCode,
                max(AuctionDate) AS AuctionDate, 
                sum(CollateralAmount) AS CollateralAmount
        FROM (
                SELECT 
                        dreal.RealEstateCepalKey AS RealEstateCepalKey,
                        dcol.AgreementKey, 
                        dreg.CollateralAmountRRE AS CollateralAmount,
                        dexp.CDI AS CDI, 
                        dexp.ReferenceDate AS BusinessDate, 
                        dexp.PortfolioName,  
                        eval.ValuationDate AS LatestValuationDate,
                        auction.FinalPrice AS AuctionFinalPrice, 
                        auction.FirstOfferPrice AS AuctionFirstOfferPrice, 
                        auction.AuctionDate, 
                        auction.`Κατηγορία3_active` AS AuctionStatusCode
                FROM dcollateral dcol
                LEFT JOIN latest_valuation eval 
                        ON eval.RealEstateKey = dcol.RealEstateKey 
                INNER JOIN (
                    SELECT DISTINCT AgreementKey, CDI, ReferenceDate, PortfolioName, ExposureKey
                    FROM dexposure
                ) dexp
                    ON dexp.AgreementKey = dcol.AgreementKey 
                    AND dcol.PortfolioName = dexp.PortfolioName
                -- INNER JOIN dexposure dexp
                --         ON dexp.AgreementKey = dcol.AgreementKey 
                --         AND dcol.PortfolioName = dexp.PortfolioName 
                INNER JOIN drealestate dreal 
                        ON dcol.RealEstateKey = dreal.RealEstateKey
                LEFT JOIN auction 
                        ON auction.Cepal_Key_Final = dreal.RealEstateCepalKey
                LEFT JOIN dregexposurefields dreg
                        ON dreg.AgreementKey = dcol.AgreementKey 
                        AND dcol.PortfolioName = dreg.PortfolioName 
                        AND dexp.ExposureKey = dreg.ExposureKey
    )
        GROUP BY RealEstateCepalKey, CDI, PortfolioName, BusinessDate
""")

joined_table_props.createOrReplaceTempView("joined_table_props")

# Create Final Tables

## Create Final Table - NPL Securitization Exposures

In [0]:
npl_securitization_exposures = spark.sql(f"""
                               SELECT 
                                 CAST({referenceDate!r} AS DATE) AS BusinessDate,
                                 CAST(joined_table_exp.PortfolioName AS STRING) AS PortfolioName,
                                 CAST(joined_table_exp.AgreementKey AS STRING) AS AgreementNo,
                                 CAST(joined_table_exp.CDI AS STRING) AS CDI,
                                 CAST(joined_table_exp.PartyTaxRegistrationNumber AS STRING) AS TaxID,
                                 CAST(joined_table_exp.ExposureKey AS STRING) AS Account_ID, 
                                 CAST(joined_table_exp.OutstandingTotalBalance AS DECIMAL(19,2)) AS TotalLegalClaimEuro,
                                 CAST(joined_table_exp.GBVEuro AS DECIMAL(19,2)) AS GBVEuro, 
                                 REPLACE(CAST(joined_table_exp.ProductKey AS STRING), ';', '-') AS Product,
                                 CAST(joined_table_exp.SourceBucket AS INT) AS SourceBucket, 
                                 CAST(joined_table_exp.ExposureStatus AS STRING) AS LoanStatus,
                                 CAST(joined_table_exp.L3869_DetailedStatus AS STRING) AS L3869_DetailedStatus,
                                 CAST(joined_table_exp.ConfiscationStatus AS STRING) AS ConfiscationStatus,
                                 CAST(joined_table_exp.LegalStatus AS STRING) AS LegalStatus,
                                 CAST(joined_table_exp.Previous_LegalStatus AS STRING) AS Previous_LegalStatus,
                                 CAST(joined_table_exp.ModificationType AS STRING) AS ModificationType,
                                 CAST(joined_table_exp.IsActiveModification AS INT) AS IsActiveModification, 
                                 CAST(joined_table_exp.TopBookBorrowerFlag AS INT) AS TopBookBorrowerFlag,
                                 CAST(joined_table_exp.IsActiveFlag AS BOOLEAN) AS IsActiveFlag,
                                 current_timestamp() as IngestionTimeStamp 
                               FROM joined_table_exp
                               """)

In [0]:
npl_sec_exposures_target_table = f"`{env_prefix}-cdh-delivery`.internalreports.nplsecuritizationexposures"

spark.sql(f"""DELETE FROM {npl_sec_exposures_target_table} WHERE BusinessDate = '{referenceDate}'""")

(npl_securitization_exposures.write
 .format("delta")
 #.option("overwriteSchema", "true")
 .mode("append")
 .saveAsTable(npl_sec_exposures_target_table))

## Create Final Table - NPL Securitization Properties

In [0]:
npl_securitization_properties = spark.sql(f"""
                                            SELECT 
                                                CAST(BusinessDate AS DATE) AS BusinessDate,
                                                CAST(RealEstateCepalKey AS STRING) AS Security_ID,
                                                CAST(CDI AS STRING) AS CDI,
                                                CAST(PortfolioName AS STRING) AS PortfolioName,
                                                CAST(LatestValuationDate AS DATE) AS CurrentValuationDate, 
                                                CAST(AuctionFirstOfferPrice AS DECIMAL(19,2)) AS JudicialValue,
                                                CAST(AuctionFinalPrice AS DECIMAL(19,2)) AS AwardPrice,
                                                CAST(AuctionStatusCode AS STRING) AS AuctionResult,
                                                CAST(AuctionDate AS DATE) AS AuctionDate,
                                                CAST(CollateralAmount AS DECIMAL(19,2)) AS CollateralAmount,
                                                current_timestamp() as IngestionTimeStamp 
                                            FROM joined_table_props
                                            """)

In [0]:
npl_sec_properties_target_table = f"`{env_prefix}-cdh-delivery`.internalreports.nplsecuritizationproperties"

spark.sql(f"""DELETE FROM {npl_sec_properties_target_table} WHERE BusinessDate = '{referenceDate}'""")

(npl_securitization_properties.write
 .format("delta")
 #.option("overwriteSchema", "true")
 .mode("append")
 .saveAsTable(npl_sec_properties_target_table))


In [0]:
from export_utils import export_data
from helpers import archive_existing_files
from pyspark.sql import functions as F
from io import BytesIO
import pandas as pd


df_npl_exp = (
    spark.table(npl_sec_properties_target_table)
         .filter(F.col("BusinessDate") == referenceDate))

pd_npl_exp = df_npl_exp.toPandas()

output_buffer = BytesIO()
with pd.ExcelWriter(output_buffer) as writer:
    pd_npl_exp.to_excel(writer, sheet_name=f"NPL_Securitization_Exposures", index=False)

output_path = f"{report_output}{output_file_prefix}Exposures_{referenceDate}.xlsx"

# Archive existing files before writing new export
os.makedirs(report_output, exist_ok=True)
archive_existing_files(report_output)

with open(output_path, "wb") as f:
    f.write(output_buffer.getvalue())

In [0]:
from export_utils import export_data
from helpers import archive_existing_files
from pyspark.sql import functions as F
from io import BytesIO
import pandas as pd


df_npl_prop = (
    spark.table(npl_sec_properties_target_table)
         .filter(F.col("BusinessDate") == referenceDate)
)

pd_npl_prop = df_npl_prop.toPandas()

output_buffer = BytesIO()
with pd.ExcelWriter(output_buffer) as writer:
    pd_npl_prop.to_excel(writer, sheet_name=f"NPL_Securitization_Properties", index=False)

output_path = f"{report_output}{output_file_prefix}Properties_{referenceDate}.xlsx"

# # Archive existing files before writing new export
# os.makedirs(report_output, exist_ok=True)
# archive_existing_files(report_output)

with open(output_path, "wb") as f:
    f.write(output_buffer.getvalue())